# Module 06 — Lesson 04
# Encoding Categorical Variables (Label, One-Hot)

---

> The numbers in this dataset are finally trustworthy after Lessons 01-03. But two columns — `department` and `performance_band` — are still text. Every model you'll build starting in Module 10 does arithmetic under the hood: it can't multiply a learned weight by `"Engineering"`. Turning text categories into numbers sounds like a trivial lookup. Done carelessly, it quietly teaches the model a relationship that was never in the data.

---
## 🎯 Learning Objectives

By the end of this lesson, you will be able to:

- Distinguish **nominal** categories (no natural order) from **ordinal** categories (a real order) and explain why that distinction picks the encoding method
- Apply **ordinal encoding** with an explicit, meaningful order using `.map()`
- Apply **one-hot encoding** with `pd.get_dummies()` for nominal categories
- Explain why naively label-encoding a nominal column invents a false numeric relationship between categories
- Recognize the **dummy variable trap** and when `drop_first=True` matters

In [1]:
import pandas as pd

employees = pd.read_csv("data/employees_clean.csv")
print(f"employees.shape: {employees.shape}")
employees[["employee_id", "name", "department", "performance_band"]].head(10)

employees.shape: (49, 7)


,employee_id,name,department,performance_band
0,1001,Rohan Pillai,HR,Exceeds Expectations
1,1002,Riya Singh,Engineering,Outstanding
2,1003,Karan Kapoor,Engineering,Meets Expectations
3,1004,Rohan Bose,Marketing,Outstanding
4,1005,Kabir Verma,HR,NaN
5,1007,Suresh Pillai,Engineering,Outstanding
6,1008,Tanvi Pillai,Engineering,Exceeds Expectations
7,1009,Pooja Gupta,Sales,Exceeds Expectations
8,1010,Simran Das,Engineering,NaN
9,1012,Sneha Bhat,Engineering,Exceeds Expectations


---
## 1. Nominal vs. Ordinal — the Decision That Picks the Method

Look at the two categorical columns in this dataset:

- **`department`** — Engineering, Sales, HR, Marketing, Finance. There's no sense in which Engineering is "more" or "less" than Sales. These are **nominal** categories: named groups with no order.
- **`performance_band`** — Needs Improvement, Meets Expectations, Exceeds Expectations, Outstanding. This one has a real order baked into the meaning: Outstanding is unambiguously better than Meets Expectations. These are **ordinal** categories.

That single distinction decides which encoding is correct:

| Type | Example here | Correct approach | Why |
|---|---|---|---|
| Ordinal | `performance_band` | Encode with an explicit order (0, 1, 2, 3, ...) | The order is real information a model can use |
| Nominal | `department` | One-hot encode (a separate 0/1 column per category) | Any single number implies an order that isn't there |

---
## 2. Ordinal Encoding: `performance_band`

Map each category to a number by hand, in the order that reflects reality — not alphabetically, not by whatever order `pandas` happens to encounter the values. `NaN` (employees too new to be rated, from Lesson 01) is left as `NaN`, not silently mapped to 0 — that would falsely claim "Needs Improvement" for someone who was never reviewed at all.

In [2]:
performance_order = {
    "Needs Improvement": 0,
    "Meets Expectations": 1,
    "Exceeds Expectations": 2,
    "Outstanding": 3,
}

employees["performance_band_encoded"] = employees["performance_band"].map(performance_order)
employees[["employee_id", "performance_band", "performance_band_encoded"]].head(10)

,employee_id,performance_band,performance_band_encoded
0,1001,Exceeds Expectations,2.0
1,1002,Outstanding,3.0
2,1003,Meets Expectations,1.0
3,1004,Outstanding,3.0
4,1005,NaN,NaN
5,1007,Outstanding,3.0
6,1008,Exceeds Expectations,2.0
7,1009,Exceeds Expectations,2.0
8,1010,NaN,NaN
9,1012,Exceeds Expectations,2.0


In [3]:
print("Rows still NaN after mapping (never reviewed -- correctly left unmapped, not zeroed):")
print(employees["performance_band_encoded"].isna().sum())

Rows still NaN after mapping (never reviewed -- correctly left unmapped, not zeroed):
8


---
## 3. Why NOT to Label-Encode a Nominal Column

Pandas can auto-assign a number per category with `.astype('category').cat.codes` — no ordering logic, just whatever order it encounters (usually alphabetical). Watch what that does to `department`.

In [4]:
naive_encoded = employees["department"].astype("category").cat.codes
pd.DataFrame({
    "department": employees["department"],
    "naive_label": naive_encoded
}).drop_duplicates().sort_values("naive_label")

,department,naive_label
1,Engineering,0
10,Finance,1
0,HR,2
3,Marketing,3
7,Sales,4


This assigns `Engineering = 0`, `Finance = 1`, `HR = 2`, `Marketing = 3`, `Sales = 4` — purely alphabetical. A model reading this column sees numbers, and numbers imply arithmetic: it will learn things like "HR is between Finance and Marketing" or "Sales minus Engineering equals 4" — relationships that are alphabetical accidents, not facts about departments. This is exactly the wrong tool for a nominal column.

---
## 4. One-Hot Encoding: `department`

One-hot encoding replaces one text column with **one 0/1 column per category** — a department either is Engineering (1) or isn't (0), with no implied ranking between columns.

In [5]:
department_dummies = pd.get_dummies(employees["department"], prefix="dept")
department_dummies.head(10)

,dept_Engineering,dept_Finance,dept_HR,dept_Marketing,dept_Sales
0,False,False,True,False,False
1,True,False,False,False,False
2,True,False,False,False,False
3,False,False,False,True,False
4,False,False,True,False,False
5,True,False,False,False,False
6,True,False,False,False,False
7,False,False,False,False,True
8,True,False,False,False,False
9,True,False,False,False,False


In [6]:
employees_encoded = pd.concat([employees, department_dummies], axis=1)
employees_encoded[["employee_id", "department", "dept_Engineering", "dept_Sales", "dept_HR"]].head(10)

,employee_id,department,dept_Engineering,dept_Sales,dept_HR
0,1001,HR,False,False,True
1,1002,Engineering,True,False,False
2,1003,Engineering,True,False,False
3,1004,Marketing,False,False,False
4,1005,HR,False,False,True
5,1007,Engineering,True,False,False
6,1008,Engineering,True,False,False
7,1009,Sales,False,True,False
8,1010,Engineering,True,False,False
9,1012,Engineering,True,False,False


---
## 5. The Dummy Variable Trap

With 5 departments, only 4 dummy columns are actually needed — if you know a row is 0 in Engineering, Finance, HR, and Marketing, it *must* be Sales. Keeping all 5 columns makes them perfectly predictable from each other (a form of multicollinearity called the **dummy variable trap**), which breaks the math behind linear models like linear/logistic regression. `pd.get_dummies(..., drop_first=True)` drops one category as the implicit "baseline".

In [7]:
department_dummies_safe = pd.get_dummies(employees["department"], prefix="dept", drop_first=True)
print(f"Columns with drop_first=False: {list(department_dummies.columns)}")
print(f"Columns with drop_first=True:  {list(department_dummies_safe.columns)}")
print()
print("An employee with all four dummy columns == 0 is implicitly Engineering (the dropped baseline).")

Columns with drop_first=False: ['dept_Engineering', 'dept_Finance', 'dept_HR', 'dept_Marketing', 'dept_Sales']
Columns with drop_first=True:  ['dept_Finance', 'dept_HR', 'dept_Marketing', 'dept_Sales']

An employee with all four dummy columns == 0 is implicitly Engineering (the dropped baseline).


**Note:** this only matters for models sensitive to multicollinearity (linear/logistic regression). Tree-based models (Module 12 onward) generally don't care and `drop_first=False` is fine, sometimes even easier to interpret. When in doubt for a linear model, drop one.

---
## ⚠️ Common Mistakes

In [8]:
# -- Mistake 1: Label-encoding a nominal column and treating the numbers as meaningful -------

print("Mistake 1 — averaging naive label codes across departments produces a number")
print("that looks meaningful but describes nothing real:")
fake_avg = naive_encoded.mean()
print(f"  Mean of department label codes: {fake_avg:.2f}")
print("  There is no such thing as an 'average department' -- this number is nonsense,")
print("  a symptom of encoding a nominal column as if it had order.")

Mistake 1 — averaging naive label codes across departments produces a number
that looks meaningful but describes nothing real:
  Mean of department label codes: 1.55
  There is no such thing as an 'average department' -- this number is nonsense,
  a symptom of encoding a nominal column as if it had order.


In [9]:
# -- Mistake 2: Mapping an ordinal column's missing category to 0 instead of leaving it NaN --

wrong_map = employees["performance_band"].fillna("Needs Improvement").map(performance_order)
right_map = employees["performance_band"].map(performance_order)

print("Mistake 2 — filling missing performance_band with the LOWEST category before mapping")
print("invents a negative review for employees who were simply never reviewed:")
print(f"  'Needs Improvement' count if NaN treated as lowest tier: "
      f"{(wrong_map == 0).sum()}")
print(f"  Actual 'Needs Improvement' count (NaN correctly excluded): "
      f"{(right_map == 0).sum()}")
print("  -> Missing and 'lowest category' are different facts. Don't collapse them.")

Mistake 2 — filling missing performance_band with the LOWEST category before mapping
invents a negative review for employees who were simply never reviewed:
  'Needs Improvement' count if NaN treated as lowest tier: 9
  Actual 'Needs Improvement' count (NaN correctly excluded): 1
  -> Missing and 'lowest category' are different facts. Don't collapse them.


In [10]:
# -- Mistake 3: One-hot encoding a high-cardinality column without thinking about column count --

print("Mistake 3 — one-hot encoding works cleanly here because department only has 5")
print("categories. A column like 'city' or 'employee_id' with hundreds of unique values would")
print(f"explode into hundreds of new columns:")
print(f"  department has {employees['department'].nunique()} unique values -> "
      f"{employees['department'].nunique()} dummy columns (fine)")
print(f"  employee_id has {employees['employee_id'].nunique()} unique values -> "
      f"{employees['employee_id'].nunique()} dummy columns if you made this mistake (not fine)")
print("  -> One-hot encoding is for LOW-cardinality nominal columns, not identifiers.")

Mistake 3 — one-hot encoding works cleanly here because department only has 5
categories. A column like 'city' or 'employee_id' with hundreds of unique values would
explode into hundreds of new columns:
  department has 5 unique values -> 5 dummy columns (fine)
  employee_id has 49 unique values -> 49 dummy columns if you made this mistake (not fine)
  -> One-hot encoding is for LOW-cardinality nominal columns, not identifiers.


---
## ✏️ Practice Exercises

### Exercise 1 — Starter: Count Categories

Load `data/employees_clean.csv` fresh. Print the unique values and counts for `department` and `performance_band` using `.value_counts(dropna=False)`.

In [11]:
# Your code here

### Exercise 2 — Ordinal Encode by Hand

Without looking back at Section 2, write your own `performance_order` dictionary and use `.map()` to create a `performance_band_encoded` column. Confirm `NaN` values stay `NaN`.

In [12]:
# Your code here

### Exercise 3 — One-Hot Encode Department

Use `pd.get_dummies()` on `department` with `drop_first=True`. Print the resulting columns and confirm the total column count is one fewer than the number of unique departments.

In [13]:
# Your code here

### Exercise 4 — Spot the Trap

For the first 5 rows of your Exercise 3 result, manually check: for each row, does the pattern of 0s and 1s across the dummy columns correctly identify the department, including the dropped baseline category? Write your check as a comment.

In [14]:
# Your code here

### Exercise 5 — (Challenge) Combine Both Encodings

Build a single DataFrame `employees_ready` that has: the ordinal-encoded `performance_band`, one-hot encoded `department` (with `drop_first=True`), and every other original numeric column -- with the original text `department` and `performance_band` columns dropped. Print its shape and dtypes.

In [15]:
# Your code here

---
## 📌 Key Takeaways

- **Ask "does this category have a real order?" before picking an encoding** — ordinal columns get an explicit, meaningful number mapping; nominal columns don't get a single number at all.
- **Label-encoding a nominal column invents a false numeric relationship between categories** (alphabetical order becomes "mathematical" order to a model) — one-hot encoding avoids this by giving each category its own independent 0/1 column.
- **One-hot encoding scales with the number of categories, and can trigger the dummy variable trap for linear models** — use `drop_first=True` for linear/logistic regression, and avoid one-hot encoding high-cardinality columns like IDs altogether.